### 01 - NumPy Por Baixo dos Panos
**Documentação NumPy** 
    - **https://numpy.org/doc/**

**Objetivo**
    - Entender como o **ndarray** funciona em memória e por que operações vetorizadas são muito mais eficientes que loops Python puros.
    
**Pré-requisitos**
    - Noções de arrays e funções


**Indice:**
1. Estrutura interna do **ndarray**
2. **dtype**, **shape**, **strides** e contiguidade
3. Views vs copies
4. Broadcasting passo a passo
5. Ufuncs, vetorização e benchmarks


In [21]:
import numpy as np
import pandas as pd
import seaborn as sns

np.random.seed(42)
sns.set_theme(style='whitegrid', context='notebook')

print('NumPy:', np.__version__)
print('pandas:', pd.__version__)

NumPy: 2.4.2
pandas: 3.0.1


### 1 - Conceito do Zero: O que é um **ndarray**?

**ndarray** é um bloco de memória homogêneo + metadados:
- ponteiro para dados
- **dtype** (tipo e tamanho)
- **shape** (dimensão)
- **strides** (passo em bytes por eixo)

Isso permite operações em lote com poucas chamadas de baixo nível.

In [22]:
a = np.arange(12, dtype=np.int32).reshape(3, 4)
print(a)
print('dtype:', a.dtype)
print('shape:', a.shape)
print('strides:', a.strides)
print('itemsize:', a.itemsize)
print('nbytes:', a.nbytes)


[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
dtype: int32
shape: (3, 4)
strides: (16, 4)
itemsize: 4
nbytes: 48


### 2 - Por Baixo dos Panos: **dtype** e layout de memória

Escolha de **dtype** afeta:
- uso de memória
- precisão numérica
- velocidade em operações vetorizadas

Trade-off clássico: **float32** economiza memória e aumenta velocidade; **float64** reduz erro numérico acumulado.

In [23]:
arr32 = np.random.rand(1_000_000).astype(np.float32)
arr64 = arr32.astype(np.float64)

print('float32 nbytes:', arr32.nbytes)
print('float64 nbytes:', arr64.nbytes)
print('Razão de memória (64/32):', arr64.nbytes / arr32.nbytes)


float32 nbytes: 4000000
float64 nbytes: 8000000
Razão de memória (64/32): 2.0


### 3 - Strides e acesso por cache

**strides** define quantos bytes avançar para andar um elemento em cada eixo. Acesso contíguo tende a aproveitar melhor cache de CPU.

In [24]:
m = np.arange(24, dtype=np.int64).reshape(4, 6)
print('C-contiguous?', m.flags['C_CONTIGUOUS'])
print('F-contiguous?', m.flags['F_CONTIGUOUS'])
print('strides:', m.strides)

m_t = m.T
print('\nTransposta')
print('C-contiguous?', m_t.flags['C_CONTIGUOUS'])
print('F-contiguous?', m_t.flags['F_CONTIGUOUS'])
print('strides:', m_t.strides)


C-contiguous? True
F-contiguous? False
strides: (48, 8)

Transposta
C-contiguous? False
F-contiguous? True
strides: (8, 48)


### 4 - Views vs Copies

Slicing simples retorna **view** (compartilha memória). Operações como fancy indexing geralmente retornam **copy**.

Erros comuns em pipelines:
- alterar view sem perceber efeito colateral
- supor que toda transformação copia dados

In [25]:
base = np.arange(10)
view = base[2:7]
copy = base[[2, 3, 4, 5, 6]]

view[0] = 999
print('base após alterar view:', base)

copy[0] = -123
print('base após alterar copy:', base)

print('view compartilha memória?', np.shares_memory(base, view))
print('copy compartilha memória?', np.shares_memory(base, copy))


base após alterar view: [  0   1 999   3   4   5   6   7   8   9]
base após alterar copy: [  0   1 999   3   4   5   6   7   8   9]
view compartilha memória? True
copy compartilha memória? False


### 5 - Broadcasting passo a passo

Broadcasting evita materializar matrizes enormes ao alinhar dimensões implicitamente.
Regras principais:
1. Comparar eixos da direita para esquerda
2. Eixos compatíveis se iguais ou um deles é 1
3. Se incompatível, erro


In [26]:
X = np.arange(12).reshape(4, 3)
b = np.array([10, 20, 30])

print('X shape:', X.shape)
print('b shape:', b.shape)
print('Resultado shape:', (X + b).shape)
print(X + b)


X shape: (4, 3)
b shape: (3,)
Resultado shape: (4, 3)
[[10 21 32]
 [13 24 35]
 [16 27 38]
 [19 30 41]]


In [27]:
A = np.random.randn(5, 1, 4)
B = np.random.randn(1, 3, 4)
C = A + B
print('A:', A.shape, 'B:', B.shape, 'C:', C.shape)


A: (5, 1, 4) B: (1, 3, 4) C: (5, 3, 4)


### 6 - Ufuncs e vetorização

Ufuncs (**np.add**, **np.exp**, etc.) executam loops otimizados em C para blocos de memória, reduzindo overhead do interpretador Python.

In [28]:
x = np.linspace(-3, 3, 5)
print('x:', x)
print('np.exp(x):', np.exp(x))
print('np.tanh(x):', np.tanh(x))


x: [-3.  -1.5  0.   1.5  3. ]
np.exp(x): [ 0.04978707  0.22313016  1.          4.48168907 20.08553692]
np.tanh(x): [-0.99505475 -0.90514825  0.          0.90514825  0.99505475]


In [ ]:
# Benchmark simples: loop Python vs NumPy vetorizado
n = 200_000_000
u = np.random.rand(n)
v = np.random.rand(n)

import time

t0 = time.perf_counter()
res_loop = np.empty(n)
for i in range(n):
    res_loop[i] = 3*u[i] + 2*v[i]
t1 = time.perf_counter()

res_vec = 3*u + 2*v
t2 = time.perf_counter()

print(f'Loop Python: {t1 - t0:.4f}s')
print(f'Vetorizado NumPy: {t2 - t1:.4f}s')

Loop Python: 39.9939s
Vetorizado NumPy: 0.4884s
Resultados próximos? True


### 7 - Operações Numéricas, Estabilidade e Overflow

Internals não são só performance; são também corretude numérica.

Exemplos de cautela:
- somas longas podem acumular erro de ponto flutuante
- **int8/int16** podem overflow silencioso
- exponenciais grandes causam overflow em **float**

In [30]:
tiny = np.array([100, 120], dtype=np.int8)
print('int8:', tiny)
print('int8 + 50:', tiny + 50)  # overflow

big = np.array([1000.0, 1001.0, 999.0])
logits = big - big.max()
probs = np.exp(logits) / np.exp(logits).sum()
print('Softmax estável:', probs)


int8: [100 120]
int8 + 50: [-106  -86]
Softmax estável: [0.24472847 0.66524096 0.09003057]


###  8 - Erros Comuns e Depuração

- Supor que reshape sempre copia (nem sempre)
- Misturar **dtype** sem perceber casting implícito
- Usar loops Python em transformações massivas
- Ignorar forma dos tensores em broadcasting